# Giorno 2 — Vehicle/People Counting & Trajectory Analysis

**Obiettivi della giornata:**
1. Conteggio veicoli con `supervision` (roboflow) su AIC21 — LineZone e PolygonZone
2. People counting su ShanghaiTech: ground truth annotations, density map, confronto con YOLO
3. Estrazione di traiettorie con `norfair` e analisi tramite heatmap

---
**Dataset:**
- **AIC21 CityFlow** — `cam_*.mp4` — telecamere di traffico urbano (AIC Track 1)
- **ShanghaiTech** — immagini statiche di folla + annotazioni ground truth (`.mat`)

**Librerie:** `supervision` (Roboflow), `norfair`, `scipy` (lettura `.mat`) + YOLOv8

> ℹ️ Anche oggi: solo inference su modelli pre-addestrati, nessun retraining.

## 0. Setup

In [ ]:
!pip install -q ultralytics supervision norfair opencv-python-headless matplotlib scipy tqdm pandas

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.io as sio
from scipy.ndimage import gaussian_filter
from pathlib import Path
from tqdm import tqdm
from IPython.display import HTML
from base64 import b64encode
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../utils')
from cv_utils import (mostra_frame, mostra_confronto, mostra_griglia,
                       stampa_info_video, estrai_frame, crea_heatmap, sovrapponi_heatmap)

def mostra_video_colab(path, width=700):
    with open(path, 'rb') as f:
        video_b64 = b64encode(f.read()).decode()
    return HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{video_b64}" type="video/mp4"></video>')

print('✅ Import completati')

In [ ]:
# ── Configurazione percorsi ──────────────────────────────────────────────────
# Se si usa Google Colab, montare il Drive e adattare DATA_DIR
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/corso_cv/data/day_2')

DATA_DIR     = Path('../data/day_2')
AICITY_DIR   = DATA_DIR / 'AICity'
SHANGHAI_DIR = DATA_DIR / 'ShangaiTech'
IMAGES_DIR   = SHANGHAI_DIR / 'images'
GT_DIR       = SHANGHAI_DIR / 'ground-truth'

# Video AIC21 CityFlow
video_files = sorted(AICITY_DIR.glob('*.mp4'))
# Immagini ShanghaiTech
shanghai_imgs = sorted(IMAGES_DIR.glob('IMG_*.jpg'))

print(f'Video AIC21 trovati: {len(video_files)}')
for v in video_files:
    print(f'  {v.name}')
print(f'\nImmagini ShanghaiTech: {len(shanghai_imgs)}')
for f in shanghai_imgs:
    print(f'  {f.name}')

In [ ]:
from ultralytics import YOLO

model_yolo = YOLO('yolov8n.pt')  # pesi già scaricati ieri
print('Modello YOLO caricato')

---
## 1. Esplorazione del dataset AIC21

**AIC21 CityFlow** è il dataset del track 1 dell'AI City Challenge 2021, che include:
- Video da telecamere stradali in condizioni diverse: normale, pioggia (`_rain`), alba (`_dawn`), neve (`_snow`)
- Stessa telecamera in condizioni diverse → ideale per testare la robustezza dei sistemi

In [ ]:
print('=' * 60)
for v in video_files:
    print(f'\n📹 {v.name}')
    stampa_info_video(str(v))
print('=' * 60)

In [ ]:
# Prima frame di ogni video
frames_preview = [estrai_frame(str(v), n=1)[0] for v in video_files]
mostra_griglia(frames_preview, [v.stem for v in video_files], colonne=4)

In [ ]:
# Selezioniamo un video base come riferimento per il counting
# cam_1 = telecamera standard senza condizioni meteo avverse
VIDEO_AIC21 = str(AICITY_DIR / 'cam_1.mp4')
print(f'Video principale: {Path(VIDEO_AIC21).name}')
stampa_info_video(VIDEO_AIC21)

---
## 2. Vehicle Counting con Supervision

**`supervision`** (Roboflow) è una libreria Python che semplifica l'annotazione e l'analisi di output YOLO.

La funzione chiave che useremo è `LineZone`: una **linea virtuale** che conta ogni oggetto che la attraversa in una direzione specifica.

**Flusso di lavoro:**
```
Frame video → YOLO detection → supervision annotators → LineZone crossing → conteggio
```

In [ ]:
import supervision as sv

print(f'supervision version: {sv.__version__}')

In [ ]:
# ── 2.1 Definizione della linea di conteggio ──────────────────────────────────
frame_cf = estrai_frame(VIDEO_AIC21, n=1)[0]
H, W = frame_cf.shape[:2]
print(f'Risoluzione: {W}x{H}')

# Linea orizzontale al 50% dell'altezza
# Modificate START e END per adattarla al vostro video
LINEA_START = sv.Point(0, H // 2)
LINEA_END   = sv.Point(W, H // 2)

frame_con_linea = frame_cf.copy()
cv2.line(frame_con_linea,
         (LINEA_START.x, LINEA_START.y),
         (LINEA_END.x, LINEA_END.y),
         (0, 255, 255), 3)
cv2.putText(frame_con_linea, 'Linea di conteggio',
            (10, H // 2 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
mostra_frame(frame_con_linea, 'Posizione della linea di conteggio')

In [ ]:
# ── 2.2 Setup supervision ─────────────────────────────────────────────────────
box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=30)
line_zone       = sv.LineZone(start=LINEA_START, end=LINEA_END)
line_zone_ann   = sv.LineZoneAnnotator(thickness=3, text_scale=1.2)
byte_tracker    = sv.ByteTrack()

print('✅ Supervision configurato')

In [ ]:
# ── 2.3 Conteggio veicoli su sequenza video ───────────────────────────────────
N_FRAME_CF = 200
OUTPUT_VIDEO_CF = '/tmp/vehicle_counting.mp4'

cap = cv2.VideoCapture(VIDEO_AIC21)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(OUTPUT_VIDEO_CF, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

conteggio_su, conteggio_giu = 0, 0

for i in tqdm(range(N_FRAME_CF), desc='Vehicle counting'):
    ret, frame = cap.read()
    if not ret:
        break

    result     = model_yolo(frame, verbose=False, classes=[2, 5, 7])[0]
    detections = sv.Detections.from_ultralytics(result)
    detections = byte_tracker.update_with_detections(detections)

    crossed_in, crossed_out = line_zone.trigger(detections)
    conteggio_su  += crossed_in.sum()
    conteggio_giu += crossed_out.sum()

    labels = [f'ID:{tid}' for tid in detections.tracker_id] if detections.tracker_id is not None else []
    frame = box_annotator.annotate(frame, detections)
    frame = label_annotator.annotate(frame, detections, labels)
    frame = trace_annotator.annotate(frame, detections)
    frame = line_zone_ann.annotate(frame, line_zone)
    cv2.putText(frame, f'↑ {conteggio_su}  ↓ {conteggio_giu}  TOT: {conteggio_su+conteggio_giu}',
                (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)
    writer.write(frame)

cap.release()
writer.release()

print(f'\n📊 Risultati:')
print(f'   ↑ {conteggio_su}  ↓ {conteggio_giu}  Totale: {conteggio_su+conteggio_giu}')

In [ ]:
mostra_video_colab(OUTPUT_VIDEO_CF)

### 2.4 PolygonZone — conteggio in aree specifiche

`supervision` permette anche di definire **aree poligonali** e contare solo gli oggetti al loro interno.
Utile per: monitorare una corsia, un parcheggio, escludere zone non rilevanti.

In [ ]:
frame_ref = estrai_frame(VIDEO_AIC21, n=1)[0]
H_r, W_r = frame_ref.shape[:2]

# Zona centrale del frame — modificate le coordinate in base alla scena
ZONA_POLYGON = np.array([
    [W_r // 4,     H_r // 4],
    [3 * W_r // 4, H_r // 4],
    [3 * W_r // 4, 3 * H_r // 4],
    [W_r // 4,     3 * H_r // 4],
])

polygon_zone = sv.PolygonZone(polygon=ZONA_POLYGON)
polygon_ann  = sv.PolygonZoneAnnotator(zone=polygon_zone, color=sv.Color.GREEN, thickness=3)

frame_zona = polygon_ann.annotate(frame_ref.copy())
mostra_frame(frame_zona, 'Zona di interesse (PolygonZone)')

In [ ]:
N_FRAME_ZONA = 100
cap = cv2.VideoCapture(VIDEO_AIC21)
conteggio_zona = []

for _ in tqdm(range(N_FRAME_ZONA), desc='Counting in zona'):
    ret, frame = cap.read()
    if not ret:
        break
    result = model_yolo(frame, verbose=False, classes=[2, 5, 7])[0]
    dets   = sv.Detections.from_ultralytics(result)
    mask   = polygon_zone.trigger(detections=dets)
    conteggio_zona.append(mask.sum())

cap.release()

plt.figure(figsize=(12, 3))
plt.plot(conteggio_zona, color='green', linewidth=1.5)
plt.fill_between(range(len(conteggio_zona)), conteggio_zona, alpha=0.3, color='green')
plt.xlabel('Frame')
plt.ylabel('Veicoli nella zona')
plt.title('Occupazione della zona nel tempo')
plt.tight_layout()
plt.show()
print(f'Media veicoli in zona: {np.mean(conteggio_zona):.1f}')

### 2.5 Robustezza alle condizioni meteo

AIC21 fornisce lo stesso incrocio in diverse condizioni atmosferiche.
Confrontiamo quante detection produce YOLO su `cam_1` (normale) vs `cam_1_rain` (pioggia) vs `cam_1_dawn` (alba).

In [ ]:
# Confronto frame singolo: normale vs pioggia vs alba
confronti_meteo = []
for nome in ['cam_1', 'cam_1_rain', 'cam_1_dawn']:
    p = AICITY_DIR / f'{nome}.mp4'
    if p.exists():
        f = estrai_frame(str(p), n=1)[0]
        ris = model_yolo(f, verbose=False, classes=[2, 5, 7])
        n = sum(1 for b in ris[0].boxes if float(b.conf[0]) > 0.3)
        ann = f.copy()
        for box in ris[0].boxes:
            if float(box.conf[0]) > 0.3:
                x1,y1,x2,y2 = map(int, box.xyxy[0])
                cv2.rectangle(ann, (x1,y1), (x2,y2), (0,255,0), 2)
        confronti_meteo.append((ann, f'{nome} ({n} det.)'))

if confronti_meteo:
    mostra_griglia([c[0] for c in confronti_meteo],
                   [c[1] for c in confronti_meteo], colonne=3)

### 🔍 Domanda
Come cambia il numero di detection al variare delle condizioni meteo?
Come gestireste un sistema in produzione esposto a queste variazioni?

### 2.6 Analisi sistematica: impatto delle condizioni meteo

Finora abbiamo confrontato 3 video su un frame singolo. Estendiamo l'analisi a **tutti i 13 video AIC21**, calcolando la media su 50 frame per ciascuno, e aggreghiamo i risultati per condizione atmosferica.

In [ ]:
# ── Detection sistematica su tutti i video AIC21 ─────────────────────────────
# ⏳ ~2–3 minuti su CPU (13 video × 50 frame)

def estrai_condizione(nome: str) -> str:
    n = Path(nome).stem
    for cond in ['rain', 'dawn', 'snow']:
        if f'_{cond}' in n:
            return cond
    return 'normale'

N_FRAME_METEO = 50
risultati_meteo = []

for v in tqdm(sorted(AICITY_DIR.glob('*.mp4')), desc='Analisi meteo'):
    cond = estrai_condizione(v.name)
    cap = cv2.VideoCapture(str(v))
    det_per_frame = []
    for _ in range(N_FRAME_METEO):
        ret, frame = cap.read()
        if not ret:
            break
        ris = model_yolo(frame, verbose=False, classes=[2, 5, 7])
        n = sum(1 for b in ris[0].boxes if float(b.conf[0]) > 0.3)
        det_per_frame.append(n)
    cap.release()
    risultati_meteo.append({
        'video':      v.stem,
        'condizione': cond,
        'media_det':  np.mean(det_per_frame) if det_per_frame else 0,
    })

df_meteo = pd.DataFrame(risultati_meteo)
print('Media detection per condizione meteo:')
print(df_meteo.groupby('condizione')['media_det'].agg(['mean', 'count', 'std']).round(2))

In [ ]:
# ── Visualizzazione ───────────────────────────────────────────────────────────
colori_meteo = {'normale': 'steelblue', 'rain': 'slategray', 'dawn': 'darkorange', 'snow': 'lightblue'}

riepilogo = df_meteo.groupby('condizione')['media_det'].agg(['mean', 'std'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for cond, row in riepilogo.iterrows():
    ax.bar(cond, row['mean'], yerr=row['std'],
           color=colori_meteo.get(cond, 'gray'), alpha=0.85, capsize=5, edgecolor='white')
ax.set_ylabel('Media detection/frame (veicoli)')
ax.set_title('Impatto condizioni meteo — detection YOLO')
ax.set_xlabel('Condizione atmosferica')

ax = axes[1]
for _, row in df_meteo.sort_values('condizione').iterrows():
    ax.bar(row['video'], row['media_det'],
           color=colori_meteo.get(row['condizione'], 'gray'), alpha=0.85)
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Media detection/frame')
ax.set_title('Dettaglio per singolo video')

from matplotlib.patches import Patch
legenda = [Patch(color=v, label=k) for k, v in colori_meteo.items()]
ax.legend(handles=legenda, fontsize=8)

plt.tight_layout()
plt.show()

---
## 3. People Counting su ShanghaiTech

### 3.1 Il dataset e le annotazioni `.mat`

**ShanghaiTech** è un benchmark classico per il **crowd counting** (conteggio di folla).
A differenza dei dataset di tracking, qui le annotazioni non sono bounding box ma **punti testa**:
ogni persona è annotata con il centro della sua testa `(x, y)`.

I file `.mat` (formato MATLAB) contengono:
- `image_info.location` — array `(N, 2)` con le coordinate `(x, y)` delle teste
- `image_info.number` — conteggio totale delle persone

**Perché usare punti invece di bounding box?**
In scene molto affollate, i bounding box si sovrappongono e diventano difficili da annotare.
I punti testa sono più veloci da annotare e sufficienti per il conteggio.

In [ ]:
def carica_mat(img_nome: str) -> dict:
    """
    Carica l'annotazione ground truth per un'immagine ShanghaiTech.
    
    Args:
        img_nome: nome file immagine es. 'IMG_3.jpg'
    Returns:
        dict con 'n_persone' (int) e 'coordinate' (array Nx2)
    """
    num = Path(img_nome).stem.replace('IMG_', '')  # '3' da 'IMG_3'
    mat_path = GT_DIR / f'GT_IMG_{num}.mat'
    mat = sio.loadmat(str(mat_path))
    info = mat['image_info'][0, 0]
    coords = info['location'][0, 0]       # shape (N, 2): colonne = [x, y]
    n = int(info['number'][0, 0].flat[0])
    return {'n_persone': n, 'coordinate': coords}


# Test su un'immagine
esempio = carica_mat('IMG_3.jpg')
print(f'Persone annotate: {esempio["n_persone"]}')
print(f'Coordinate shape: {esempio["coordinate"].shape}')
print(f'Prime 3 annotazioni (x, y): {esempio["coordinate"][:3]}')

In [ ]:
# ── 3.2 Panoramica del subset ─────────────────────────────────────────────────
import pandas as pd

righe = []
for img_path in shanghai_imgs:
    gt = carica_mat(img_path.name)
    righe.append({'immagine': img_path.name, 'n_persone_GT': gt['n_persone']})

df_gt = pd.DataFrame(righe).sort_values('n_persone_GT')
print(df_gt.to_string(index=False))
print(f'\nMedia: {df_gt["n_persone_GT"].mean():.0f}  |  Min: {df_gt["n_persone_GT"].min()}  |  Max: {df_gt["n_persone_GT"].max()}')

### 3.3 Visualizzazione delle annotazioni ground truth

Visualizziamo i punti testa annotati sull'immagine originale. Ogni punto corrisponde a una persona.

In [ ]:
def visualizza_gt(img_path: str, raggio: int = 5, colore=(0, 255, 0)) -> np.ndarray:
    """
    Disegna i punti testa GT sull'immagine.
    """
    img = cv2.imread(str(img_path))
    gt  = carica_mat(Path(img_path).name)
    for (x, y) in gt['coordinate']:
        cv2.circle(img, (int(x), int(y)), raggio, colore, -1)
    return img


# Visualizzazione sulle prime 4 immagini del subset
frames_gt = []
titoli_gt = []
for img_path in shanghai_imgs[:4]:
    gt = carica_mat(img_path.name)
    ann = visualizza_gt(img_path)
    frames_gt.append(ann)
    titoli_gt.append(f'{img_path.stem}\n{gt["n_persone"]} persone (GT)')

mostra_griglia(frames_gt, titoli_gt, colonne=2)

### 3.4 Density Map — dalla ground truth alla "mappa di calore delle presenze"

La **density map** è il concetto centrale del crowd counting moderno.
Trasforma i punti annotati in un'immagine continua dove:
- ogni punto testa viene sostituito da una **gaussiana 2D** centrata su quel punto
- la somma della density map = numero totale di persone
- le zone dense appaiono con valori più alti

I modelli di crowd counting allo stato dell'arte (CSRNet, MCNN, BayesCrowd...) sono addestrati a
**predire la density map** direttamente dall'immagine, invece di rilevare singole persone.

Noi la calcoliamo dalla ground truth per capire come "dovrebbe apparire" la predizione ideale.

In [ ]:
def crea_density_map(img_shape: tuple, coordinate: np.ndarray,
                      sigma: float = 15.0) -> np.ndarray:
    """
    Genera la density map da annotazioni puntuali.
    
    Args:
        img_shape: (H, W) dell'immagine
        coordinate: array (N, 2) con colonne (x, y)
        sigma: deviazione standard della gaussiana (px)
    Returns:
        density map normalizzata, shape (H, W)
    """
    H, W = img_shape[:2]
    dm = np.zeros((H, W), dtype=np.float32)

    for (x, y) in coordinate:
        xi, yi = int(np.clip(x, 0, W-1)), int(np.clip(y, 0, H-1))
        dm[yi, xi] += 1.0

    dm = gaussian_filter(dm, sigma=sigma)
    return dm

In [ ]:
# ── Density map su un'immagine con alta densità ────────────────────────────────
# Scegliamo l'immagine con più persone nel subset
img_piu_densa = max(shanghai_imgs,
                     key=lambda p: carica_mat(p.name)['n_persone'])
gt_densa = carica_mat(img_piu_densa.name)
img_bgr  = cv2.imread(str(img_piu_densa))
H_sh, W_sh = img_bgr.shape[:2]

print(f'Immagine: {img_piu_densa.name}')
print(f'Persone GT: {gt_densa["n_persone"]}')

dm = crea_density_map((H_sh, W_sh), gt_densa['coordinate'], sigma=15)

# Visualizzazione affiancata
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1) Immagine originale
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title('Immagine originale', fontsize=12)
axes[0].axis('off')

# 2) Annotazioni GT (punti testa)
img_punti = img_bgr.copy()
for (x, y) in gt_densa['coordinate']:
    cv2.circle(img_punti, (int(x), int(y)), 4, (0, 255, 0), -1)
axes[1].imshow(cv2.cvtColor(img_punti, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Punti GT ({gt_densa["n_persone"]} persone)', fontsize=12)
axes[1].axis('off')

# 3) Density map
axes[2].imshow(dm, cmap='jet', interpolation='bilinear')
axes[2].set_title(f'Density map (σ=15)\nIntegrale ≈ {dm.sum():.1f}', fontsize=12)
axes[2].axis('off')
plt.colorbar(axes[2].images[0], ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle(img_piu_densa.name, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Somma density map: {dm.sum():.1f}  (atteso: {gt_densa["n_persone"]})')

In [ ]:
# ── Proprietà fondamentale: integrale costante al variare di σ ────────────────
# La somma della density map deve essere uguale al numero di persone
# per qualsiasi valore di sigma — dimostriamo questa proprietà matematica.

sigmas_test = [1, 3, 5, 10, 15, 20, 30, 50, 80, 120]
somme_test  = []

for s in sigmas_test:
    dm_s = crea_density_map((H_sh, W_sh), gt_densa['coordinate'], sigma=s)
    somme_test.append(dm_s.sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(sigmas_test, somme_test, 'o-', color='steelblue', linewidth=2, markersize=7)
ax.axhline(gt_densa['n_persone'], color='red', linestyle='--', linewidth=1.5,
           label=f'GT: {gt_densa["n_persone"]} persone')
ax.set_xlabel('Sigma (σ) [pixel]')
ax.set_ylabel('Integrale density map')
ax.set_title('Integrale al variare di σ')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
errore_rel = [abs(s - gt_densa['n_persone']) / gt_densa['n_persone'] * 100
              for s in somme_test]
ax.plot(sigmas_test, errore_rel, 's-', color='coral', linewidth=2, markersize=7)
ax.set_xlabel('Sigma (σ) [pixel]')
ax.set_ylabel('Errore rispetto a GT (%)')
ax.set_title('Errore integrale (dovrebbe essere ≈ 0)')
ax.grid(True, alpha=0.3)

plt.suptitle('La somma della density map rimane costante ∀σ (proprietà gaussiana)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'GT: {gt_densa["n_persone"]} persone')
print(f'Somma min (σ={sigmas_test[int(np.argmin(somme_test))]}): {min(somme_test):.2f}')
print(f'Somma max (σ={sigmas_test[int(np.argmax(somme_test))]}): {max(somme_test):.2f}')
print("→ L'integrale è stabile indipendentemente da σ ✅")

In [ ]:
# ── Effetto di sigma sulla density map ────────────────────────────────────────
# Sigma piccolo → punti precisi; Sigma grande → blob diffusi
img_test = cv2.imread(str(img_piu_densa))
gt_test  = carica_mat(img_piu_densa.name)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, sigma in zip(axes, [5, 15, 30, 60]):
    dm_s = crea_density_map(img_test.shape, gt_test['coordinate'], sigma=sigma)
    ax.imshow(dm_s, cmap='hot', interpolation='bilinear')
    ax.set_title(f'σ = {sigma}\nSomma = {dm_s.sum():.1f}', fontsize=11)
    ax.axis('off')

plt.suptitle('Effetto di sigma sulla density map', fontsize=13)
plt.tight_layout()
plt.show()

### 3.7 Confronto esplicito: bassa vs alta densità

Su quale range di densità YOLO funziona meglio? Confrontiamo l'immagine con **meno persone** e quella con **più persone** del subset, visualizzando per ciascuna: punti GT, detection YOLO, density map.

### 3.5 Confronto: YOLO detection vs Ground Truth

Applichiamo YOLO a ogni immagine e confrontiamo il conteggio con il ground truth.
Questo mostra quantitativamente **perché YOLO non è adatto al crowd counting su folla densa**.

In [ ]:
# ── Confronto esplicito: immagine a bassa vs alta densità ─────────────────────
img_minima = df.iloc[0]   # df ordinato per GT crescente → meno persone
img_massima = df.iloc[-1]  # → più persone

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row_idx, (row_df, label) in enumerate([(img_minima, 'BASSA densità'),
                                            (img_massima, 'ALTA densità')]):
    nome_img = row_df['immagine']
    gt       = carica_mat(nome_img)
    img      = cv2.imread(str(IMAGES_DIR / nome_img))
    H_img, W_img = img.shape[:2]

    # Col 1: GT punti testa
    img_gt = img.copy()
    for (x, y) in gt['coordinate']:
        cv2.circle(img_gt, (int(x), int(y)), 4, (0, 255, 0), -1)
    axes[row_idx][0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB))
    axes[row_idx][0].set_title(f'{label} — {Path(nome_img).stem}\nGT: {gt["n_persone"]} persone')
    axes[row_idx][0].axis('off')

    # Col 2: YOLO detection
    ris      = model_yolo(img, verbose=False, classes=[0], conf=SOGLIA_YOLO)
    img_yolo = img.copy()
    n_yolo_v = sum(1 for b in ris[0].boxes if float(b.conf[0]) >= SOGLIA_YOLO)
    for box in ris[0].boxes:
        if float(box.conf[0]) >= SOGLIA_YOLO:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(img_yolo, (x1,y1),(x2,y2),(0,100,255),2)
    errore = abs(gt['n_persone'] - n_yolo_v) / max(gt['n_persone'], 1) * 100
    axes[row_idx][1].imshow(cv2.cvtColor(img_yolo, cv2.COLOR_BGR2RGB))
    axes[row_idx][1].set_title(f'YOLO: {n_yolo_v} detection\nerrore: {errore:.0f}%')
    axes[row_idx][1].axis('off')

    # Col 3: density map GT
    dm = crea_density_map((H_img, W_img), gt['coordinate'], sigma=15)
    axes[row_idx][2].imshow(dm, cmap='jet', interpolation='bilinear')
    axes[row_idx][2].set_title(f'Density map GT\nintegrale ≈ {dm.sum():.0f}')
    axes[row_idx][2].axis('off')

plt.suptitle('YOLO vs GT: bassa vs alta densità di folla', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Conteggio YOLO vs GT su tutte le immagini del subset ─────────────────────
SOGLIA_YOLO = 0.25  # soglia bassa per massimizzare le detection

risultati = []
for img_path in tqdm(shanghai_imgs, desc='YOLO su ShanghaiTech'):
    img = cv2.imread(str(img_path))
    gt  = carica_mat(img_path.name)

    ris = model_yolo(img, verbose=False, classes=[0], conf=SOGLIA_YOLO)
    n_yolo = sum(1 for b in ris[0].boxes if float(b.conf[0]) >= SOGLIA_YOLO)

    risultati.append({
        'immagine':   img_path.name,
        'GT':         gt['n_persone'],
        'YOLO':       n_yolo,
        'errore_abs': abs(gt['n_persone'] - n_yolo),
        'errore_pct': abs(gt['n_persone'] - n_yolo) / gt['n_persone'] * 100,
    })

df = pd.DataFrame(risultati).sort_values('GT')
print(df[['immagine','GT','YOLO','errore_abs','errore_pct']].to_string(index=False, float_format='{:.1f}'.format))
print(f'\nErrore medio: {df["errore_abs"].mean():.1f} persone  ({df["errore_pct"].mean():.1f}%)')

In [ ]:
# Grafico GT vs YOLO
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter GT vs YOLO
ax = axes[0]
ax.scatter(df['GT'], df['YOLO'], color='steelblue', alpha=0.8, s=60, zorder=3)
lim_max = max(df['GT'].max(), df['YOLO'].max()) * 1.05
ax.plot([0, lim_max], [0, lim_max], 'r--', linewidth=1.5, label='Predizione perfetta')
ax.set_xlabel('Persone GT (reale)')
ax.set_ylabel('Persone YOLO')
ax.set_title('GT vs YOLO — ShanghaiTech')
ax.legend()
ax.grid(True, alpha=0.3)
for _, row in df.iterrows():
    ax.annotate(Path(row['immagine']).stem,
                (row['GT'], row['YOLO']),
                fontsize=7, xytext=(5, 3), textcoords='offset points')

# Errore percentuale per immagine
ax = axes[1]
nomi_brevi = [Path(n).stem for n in df['immagine']]
ax.barh(nomi_brevi, df['errore_pct'], color='coral', edgecolor='white')
ax.set_xlabel('Errore percentuale (%)')
ax.set_title('Errore YOLO rispetto al GT')
ax.axvline(df['errore_pct'].mean(), color='red', linestyle='--',
           label=f'Media: {df["errore_pct"].mean():.1f}%')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualizzazione qualitativa: immagine GT vs detection YOLO ────────────────
# Selezioniamo l'immagine con più persone per il confronto più significativo

img_path_test = img_piu_densa
gt_test  = carica_mat(img_path_test.name)
img_test = cv2.imread(str(img_path_test))

# Ground truth (punti testa)
img_gt_vis = img_test.copy()
for (x, y) in gt_test['coordinate']:
    cv2.circle(img_gt_vis, (int(x), int(y)), 4, (0, 255, 0), -1)

# YOLO detection
ris_test = model_yolo(img_test, verbose=False, classes=[0], conf=SOGLIA_YOLO)
img_yolo_vis = img_test.copy()
n_yolo_test = 0
for box in ris_test[0].boxes:
    if float(box.conf[0]) >= SOGLIA_YOLO:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(img_yolo_vis, (x1,y1), (x2,y2), (0, 100, 255), 2)
        n_yolo_test += 1

# Density map GT sovrapposta
dm_vis = crea_density_map(img_test.shape, gt_test['coordinate'], sigma=15)
dm_uint8 = (dm_vis / dm_vis.max() * 255).astype(np.uint8)
dm_color = cv2.applyColorMap(dm_uint8, cv2.COLORMAP_JET)
img_dm_vis = cv2.addWeighted(img_test, 0.5, dm_color, 0.5, 0)

mostra_griglia(
    [img_gt_vis, img_yolo_vis, img_dm_vis],
    [f'GT: {gt_test["n_persone"]} persone (punti testa)',
     f'YOLO: {n_yolo_test} detection (conf≥{SOGLIA_YOLO})',
     'Density map GT'],
    colonne=3
)

### 🔍 Osservazioni
- Perché YOLO **sottostima** drasticamente il numero di persone su scene dense?
- Come funziona la density map come alternativa alle bounding box?
- Quali vantaggi avrebbe un modello addestrato a predire density map rispetto a YOLO in questo contesto?

---
## 4. Trajectory Extraction con Norfair

**Norfair** è un tracker leggero e flessibile che offre accesso diretto alla storia di ogni traccia.

**Differenza rispetto a ieri:**
- Giorno 1 → ci interessava *chi* è nel frame e *dove*
- Oggi → ci interessa *dove è andato* nel tempo → traiettorie, pattern di movimento, heatmap

In [ ]:
import norfair
from norfair import Detection, Tracker

print(f'norfair version: {norfair.__version__}')

In [ ]:
def yolo_a_norfair(boxes_yolo, soglia_conf: float = 0.3) -> list:
    """
    Converte le detection YOLO nel formato atteso da Norfair.
    Usa il centro del bounding box come punto tracciato.
    """
    detections = []
    for box in boxes_yolo:
        conf = float(box.conf[0])
        if conf < soglia_conf:
            continue
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        detections.append(
            Detection(
                points=np.array([[cx, cy]]),
                scores=np.array([conf]),
                data={'bbox': [x1, y1, x2, y2]}
            )
        )
    return detections

In [ ]:
# ── 4.1 Tracking con norfair e raccolta traiettorie ───────────────────────────
N_FRAME_TRAJ = 200
OUTPUT_TRAJ  = '/tmp/traiettorie_norfair.mp4'

tracker_norfair = Tracker(
    distance_function='euclidean',
    distance_threshold=50,
    hit_counter_max=15,
    initialization_delay=2,
)

traiettorie = defaultdict(list)

cap = cv2.VideoCapture(VIDEO_AIC21)
fps_nf = cap.get(cv2.CAP_PROP_FPS)
W_nf   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_nf   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(OUTPUT_TRAJ, cv2.VideoWriter_fourcc(*'mp4v'),
                         fps_nf, (W_nf, H_nf))

np.random.seed(42)
PALETTE_NF = np.random.randint(0, 255, size=(200, 3), dtype=np.uint8)

for i in tqdm(range(N_FRAME_TRAJ), desc='Trajectory extraction'):
    ret, frame = cap.read()
    if not ret:
        break

    ris = model_yolo(frame, verbose=False, classes=[2, 5, 7])[0]
    dets_nf = yolo_a_norfair(ris.boxes, soglia_conf=0.3)
    tracked = tracker_norfair.update(detections=dets_nf)

    out_frame = frame.copy()
    for obj in tracked:
        tid = obj.id
        cx, cy = obj.estimate[0]
        traiettorie[tid].append((int(cx), int(cy)))

        colore = tuple(int(c) for c in PALETTE_NF[tid % len(PALETTE_NF)])

        if obj.last_detection is not None and obj.last_detection.data:
            x1,y1,x2,y2 = map(int, obj.last_detection.data['bbox'])
            cv2.rectangle(out_frame, (x1,y1), (x2,y2), colore, 2)
            cv2.putText(out_frame, f'ID:{tid}', (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, colore, 2)

        punti_traj = traiettorie[tid][-30:]
        for j in range(1, len(punti_traj)):
            alpha = j / len(punti_traj)
            c_fade = tuple(int(c * alpha) for c in colore)
            cv2.line(out_frame, punti_traj[j-1], punti_traj[j], c_fade, 2)

        cv2.circle(out_frame, (int(cx), int(cy)), 5, colore, -1)

    cv2.putText(out_frame, f'Track attivi: {len(tracked)}',
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    writer.write(out_frame)

cap.release()
writer.release()

print(f'Traiettorie raccolte: {len(traiettorie)}')
print(f'Durata media: {np.mean([len(v) for v in traiettorie.values()]):.1f} frame')
mostra_video_colab(OUTPUT_TRAJ)

### 4.2 Analisi delle traiettorie

In [ ]:
# Plot traiettorie su sfondo scurito
frame_sfondo = estrai_frame(VIDEO_AIC21, n=1)[0]
sfondo = (frame_sfondo.astype(np.float32) * 0.4).astype(np.uint8)

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(cv2.cvtColor(sfondo, cv2.COLOR_BGR2RGB))

cmap_traj = cm.get_cmap('tab20')
for idx, (tid, punti) in enumerate(traiettorie.items()):
    if len(punti) < 5:
        continue
    xs = [p[0] for p in punti]
    ys = [p[1] for p in punti]
    ax.plot(xs, ys, linewidth=1.5, alpha=0.8, color=cmap_traj(idx % 20))
    ax.plot(xs[-1], ys[-1], 'o', markersize=4, color=cmap_traj(idx % 20))

ax.set_title(f'Traiettorie — {len(traiettorie)} veicoli tracciati', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap di densità del traffico
tutti_i_punti = [p for punti in traiettorie.values() for p in punti]
heatmap = crea_heatmap(tutti_i_punti, shape=(H_nf, W_nf), sigma=25)
frame_hm = sovrapponi_heatmap(estrai_frame(VIDEO_AIC21, n=1)[0], heatmap, alpha=0.6)
mostra_frame(frame_hm, 'Heatmap densità traffico — AIC21')

In [ ]:
# Distribuzione lunghezze e velocità
lunghezze = [len(v) for v in traiettorie.values()]
velocita  = []
for punti in traiettorie.values():
    if len(punti) < 2:
        continue
    dists = [np.linalg.norm(np.array(punti[i]) - np.array(punti[i-1]))
             for i in range(1, len(punti))]
    velocita.append(np.mean(dists))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(lunghezze, bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Durata (frame)')
axes[0].set_ylabel('N° oggetti')
axes[0].set_title('Distribuzione durata traiettorie')

axes[1].hist(velocita, bins=20, color='darkorange', edgecolor='white')
axes[1].set_xlabel('Velocità media (pixel/frame)')
axes[1].set_ylabel('N° oggetti')
axes[1].set_title('Distribuzione velocità media')

plt.tight_layout()
plt.show()
print(f'Durata media: {np.mean(lunghezze):.1f} frame')
print(f'Velocità media: {np.mean(velocita):.1f} pixel/frame')

---
## 5. Riepilogo e discussione

### Cosa abbiamo fatto oggi:
1. **Vehicle counting** con `supervision` LineZone e PolygonZone su AIC21
2. **Robustezza meteorologica**: confronto detection in condizioni normali / pioggia / alba
3. **ShanghaiTech**: annotazioni `.mat` → density map GT → confronto con YOLO
4. **Trajectory extraction** con norfair → heatmap, analisi velocità

### Punti chiave:
- **LineZone / PolygonZone**: strumenti potenti per il counting senza retraining
- **YOLO su folla densa**: errori tipici > 80% → necessari modelli specializzati
- **Density map**: rappresentazione alternativa alle bounding box per la folla
- **Traiettorie**: permettono analisi comportamentale non possibili con la sola detection

### 🔍 Domande di riflessione:
- Quali settori potrebbero beneficiare delle density map (aeroporti, concerti, piazze)?
- Come combinereste counting e traiettorie per rilevare eventi anomali?
- Le condizioni meteo influenzano allo stesso modo YOLO e un ipotetico modello di density map?

In [ ]:
# ── ESERCIZIO FINALE ──────────────────────────────────────────────────────────
# A) Scegliete un'immagine ShanghaiTech a bassa densità e una ad alta densità.
#    Confrontate le density map GT e le detection YOLO.
#    In quale caso YOLO è più accurato? Perché?
#
# B) Provate la LineZone su un video con condizioni meteo avverse (cam_1_rain).
#    Come cambia il conteggio rispetto alla versione normale?
#
# C) Modificate sigma nella density map e osservate come cambia l'integrale.
#    Cosa succede con sigma molto piccolo?

print('Buon lavoro! 🎉')